# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MissNaliaka/SEO-Content-Opportunity-Scoring/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*


Decision moment: `2026-04-30`. Feature window: `2026-01-30`–`2026-04-30` (90 days), split at `2026-03-15` into an early half and a late half for the decline signal. Content must exist before the window starts (`content_created_date <= 2026-01-30`) and belong to a client with coverage from the window start (`gsc_data_start <= 2026-01-30`) — otherwise a page or client could look "declining" simply because it didn't have data yet, not because it actually declined.

In [2]:
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get("PurpleElegantBass749671")
print(f"Token loaded: {hf_token[:6]}... (length {len(hf_token)})" if hf_token else "Token is empty/None!")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
DECISION_MOMENT = "2026-04-30"
WINDOW_START = "2026-01-30"
SPLIT_DATE = "2026-03-15"

#read the daily content performance data from Parquet files covering January through April 2026
#filter it to the specific feature window between WINDOW_START and DECISION_MOMENT
#group the data by client and page
#split impressions into an early period (imp_early) and a later period (imp_late) based on SPLIT_DATE
#calculates total impressions, total clicks, and average search position for the entire window
#produce a DataFrame with one row per client-page combination

feature_paths = [f"{rel}/fact_content_daily_performance/month=2026-0{m}/*.parquet" for m in [1, 2, 3, 4]]

feature_df = con.sql(f"""
    WITH daily AS (
        SELECT client_hash_id, content_hash_id, report_date,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet([{', '.join(f"'{p}'" for p in feature_paths)}])
        WHERE report_date BETWEEN DATE '{WINDOW_START}' AND DATE '{DECISION_MOMENT}'
    )


    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '{SPLIT_DATE}' THEN gsc_impressions ELSE 0 END) AS imp_early,
        SUM(CASE WHEN report_date >  DATE '{SPLIT_DATE}' THEN gsc_impressions ELSE 0 END) AS imp_late,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_90d,
        AVG(gsc_avg_position) AS avg_position_90d
    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

qualifying = con.sql(f"""
    SELECT c.content_hash_id, c.client_hash_id, c.content_type, c.main_intent,
           c.word_count, c.char_count, c.competition_level, c.search_volume
    FROM read_parquet('{rel}/dim_content.parquet') c
    JOIN read_parquet('{rel}/dim_clients.parquet') cl USING (client_hash_id)
    WHERE c.content_created_date <= DATE '{WINDOW_START}'
      AND cl.gsc_data_start <= DATE '{WINDOW_START}'
""").df()

feature_df = feature_df.merge(qualifying, on=["client_hash_id", "content_hash_id"], how="inner")
feature_df["ctr"] = np.where(
    feature_df["impressions_90d"] > 0,
    (feature_df["clicks_90d"] / feature_df["impressions_90d"]) * 100,
    np.nan,
)
feature_df["is_declining"] = (feature_df["imp_late"] < feature_df["imp_early"]).astype(int)

print(f"Rows: {len(feature_df):,}")
feature_df.head()

Token loaded: hf_IcB... (length 37)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 260,285


,client_hash_id,content_hash_id,imp_early,imp_late,impressions_90d,clicks_90d,avg_position_90d,content_type,main_intent,word_count,char_count,competition_level,search_volume,ctr,is_declining
0,client_e547b89c05043229,content_7292af5ee9096ce1,1438.0,1802.0,3240.0,2.0,35.669767,keyword article,informational,1471,9089,LOW,50,0.061728,0
1,client_e547b89c05043229,content_14a6ade39e9a8e31,824.0,1770.0,2594.0,3.0,21.947559,keyword article,transactional,2996,18548,HIGH,20,0.115652,0
2,client_e547b89c05043229,content_5ed859ae9dc2e356,0.0,0.0,0.0,0.0,NaN,keyword article,transactional,1611,9874,LOW,10,NaN,0
3,client_e547b89c05043229,content_2618be372e19730f,400.0,709.0,1109.0,0.0,28.642764,keyword article,commercial,1497,9416,MEDIUM,10,0.000000,0
4,client_e547b89c05043229,content_1da7d268111c5875,1209.0,875.0,2084.0,3.0,19.449698,keyword article,transactional,2760,17157,LOW,10,0.143954,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**imp_early, and imp_late**: These features represent the impressions in the first and second half of the 90-day window. Missing - 0 when a page genuinely got no impressions, available before decision moment: yes, both halves end by 2026-04-30.

**impressions_90d, clicks_90d**: These are full window totals obtained from fact_content_daily_performance. Missing - 0 when a page genuinely got no impressions, available before decision moment: Yes

**avg_position_90d**: Mean gsc_avg_position over the 90-day window. **NaN for 112,948 of 260,285 rows (43.4%)** — pages with zero impressions have no position to average. This is a real, large pattern (not random) — any position-based signal must filter on visibility first. Available before decision moment: Yes.

**click through rate (ctr)**: clicks_90d * impressions_90d * 100. NaN wherever `impressions_90d = 0`, same rows as the position gap. Available before decision moment: Yes.

`content_type`, `main_intent`, `word_count`, `char_count`, `competition_level`, `search_volume`: Content metadata from `dim_content. Occasional nulls, not checked in depth here — content properties, low leakage risk either way | Yes — these barely change once a page exists.

**Missingness follows a pattern, not randomness:** the same ~112,948 rows are null in both `avg_position_90d` and `ctr` — pages with zero search visibility in the window. A blind `fillna(0)` on CTR would wrongly read "zero clicks, zero impressions" as "0% CTR" (a real, if bad, number) rather than "no data to judge" — these need a `visible = impressions_90d > 0` (or a higher floor) gate before any CTR-based rule touches them, same principle as the CSV's `impression_tier == "no_data"` handling.








In [3]:

print("Missingness check:")
print(f"NaN avg_position_90d: {feature_df['avg_position_90d'].isna().sum():,} ({feature_df['avg_position_90d'].isna().mean():.1%})")
print(f"NaN ctr: {feature_df['ctr'].isna().sum():,} ({feature_df['ctr'].isna().mean():.1%})")
print(f"Rows where both are NaN together: {(feature_df['avg_position_90d'].isna() & feature_df['ctr'].isna()).sum():,}")
print(f"\nis_declining base rate: {feature_df['is_declining'].mean():.3f}")

Missingness check:
NaN avg_position_90d: 112,948 (43.4%)
NaN ctr: 112,948 (43.4%)
Rows where both are NaN together: 112,948

is_declining base rate: 0.298


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**The trap we already found and avoided, before writing any code:** `dim_content.content_updated_date` looked like a natural staleness signal (`decision_moment − content_updated_date`), the direct warehouse equivalent of the CSV's `days_since_last_update`. It's excluded here on purpose. `dim_content` is a **snapshot** table — it stores only the current, latest update date per page, not a history of edits. A quick check showed only 26.3% of all content had `content_updated_date <= decision_moment`; for the rest, the true pre-decision-moment update history is invisible — overwritten by a later edit the table no longer shows. Using it would systematically mislabel actively-maintained pages as "stale," which is backwards from what the signal claims to measure. That's why this baseline uses a **within-window decline signal** (`imp_early` vs `imp_late`) instead — built entirely from `fact_content_daily_performance`, a true daily history, not a latest-state snapshot.

**Formal leakage test, the same shape as the Week 3 trap:** fit a quick honest model on this feature set, then add a deliberately leaky column built from May data (the month *after* the decision moment) and watch the score jump.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.tree import DecisionTreeClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# check-only label: did the page keep declining into May? (May is NEVER a feature)
may_df = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_may
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-05/*.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

trap_df = feature_df.merge(may_df, on=["client_hash_id", "content_hash_id"], how="left")
trap_df["impressions_may"] = trap_df["impressions_may"].fillna(0)
late_days, may_days = 46, 31
trap_df["rate_late"] = trap_df["imp_late"] / late_days
trap_df["rate_may"] = trap_df["impressions_may"] / may_days
trap_df["still_declining_may"] = (trap_df["rate_may"] < 0.8 * trap_df["rate_late"]).astype(int)

honest_features = ["imp_early", "imp_late", "impressions_90d", "clicks_90d", "avg_position_90d"]
X_honest = trap_df[honest_features].fillna(0)
y = trap_df["still_declining_may"].values

honest_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
honest_tree.fit(X_honest, y)
honest_scores = honest_tree.predict_proba(X_honest)[:, 1]
print(f"HONEST Precision@50 (in-sample): {precision_at_k(honest_scores, y, 50):.3f}")

# spring the trap: a feature computed straight from the May label itself
trap_df["leak_col"] = trap_df["rate_may"]  # directly derived from the label
leaky_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
X_leaky = trap_df[honest_features + ["leak_col"]].fillna(0)
leaky_tree.fit(X_leaky, y)
leaky_scores = leaky_tree.predict_proba(X_leaky)[:, 1]
print(f"LEAKY Precision@50 (with leak_col): {precision_at_k(leaky_scores, y, 50):.3f}")
print("\nExpectation: leaky score should jump toward 1.0 — that's the trap working, not a good model.")
print("leak_col is dropped from the real feature set. Only imp_early/imp_late/impressions_90d/clicks_90d/avg_position_90d are used.")



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

HONEST Precision@50 (in-sample): 0.660
LEAKY Precision@50 (with leak_col): 1.000

Expectation: leaky score should jump toward 1.0 — that's the trap working, not a good model.
leak_col is dropped from the real feature set. Only imp_early/imp_late/impressions_90d/clicks_90d/avg_position_90d are used.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **`content_updated_date`, `last_optimized_date`** — snapshot-table dates that don't reliably reflect state as of the decision moment (see leakage hunt above). Using either risks systematically mislabeling actively-maintained pages.
- **`optimization_eligible_date`** — a forward-looking product/workflow field (when a page becomes eligible for *future* optimization), not an observed signal about the past.
- **`is_published`, `is_deleted`** — product/workflow state flags, not organic search signals; per the lane guide's rule to treat product decisions as background, not truth.
- **`provider_used`, `model_used`** — which LLM/provider generated the content. Interesting metadata, but not a content-performance signal, and risks becoming a proxy for "which client/workflow" rather than a real driver of decline — excluded from the rule for this pass.
- **`keyword_created_date`** — redundant with `content_created_date` for this purpose; not needed.
- **Any `fact_content_daily_performance` row after `2026-04-30`** (May, June) — used *only* as a check-only label in the leakage hunt above, never as a feature or baseline input, same convention as `trend_direction`/`trend_pct` in the CSV work.
- **`backlinks`, `category_count`, `search_volume`, `competition`/`competition_level`, `cpc`** — not excluded outright, but not used in this baseline's rule either; kept as candidate features for Week 5's model, where they can be tested properly rather than folded into a hand-set rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.